In [2]:
import numpy as np
import pandas as pd 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_absolute_error

In [3]:
df = pd.read_csv('../train_V2_cleaned.csv')

df_train = df[df["outcome_damage_inc"]!=0].copy()

df_train = df_train.drop(columns=["outcome_profit","outcome_damage_inc"])

df_train = df_train.sample(frac=1, random_state=123)

X = df_train.drop(columns="outcome_damage_amount")
y = df_train["outcome_damage_amount"]

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=1234)

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)
y_log = np.log1p(y)

best_params = { "n_estimators": 397,"learning_rate": 0.03,"max_depth": 12,"max_leaf_nodes": 32,"min_samples_split": 2,"min_samples_leaf": 16, "max_features": "log2","subsample": 0.737, "loss": "squared_error","criterion":"squared_error"}

gbm_pipeline = Pipeline([
    ("gbm", GradientBoostingRegressor(
        **best_params,
        random_state=42
    ))
])

cv_scores = cross_val_score(
    gbm_pipeline,
    X_train,
    y_train_log,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

print("CV R² scores:", cv_scores)
print("Mean CV R²:", cv_scores.mean())
print("Std CV R²:", cv_scores.std())

gbm_pipeline.fit(X_train, y_train_log)
y_pred_log = gbm_pipeline.predict(X_test)
y_pred = np.expm1(y_pred_log)

print("Test R² (log scale):", r2_score(y_test_log, y_pred_log))
print("Test MAE (original scale):", mean_absolute_error(y_test, y_pred))

CV R² scores: [0.31466084 0.2916773  0.28428441 0.32633015 0.20862277]
Mean CV R²: 0.28511509313002126
Std CV R²: 0.04114963140985894
Test R² (log scale): 0.23673887421587192
Test MAE (original scale): 268.2231375995654


# Train on all data

In [4]:
gbm_pipeline.fit(X, y_log)

df_score = pd.read_csv("../score_cleaned.csv")

df_predictions = df_score.drop(columns=[
    "outcome_damage_inc",
    "outcome_damage_amount",
    "outcome_profit",
    "outcome_damage_inc_proba"
], errors="ignore")

y_pred_log = gbm_pipeline.predict(df_predictions)
y_score_normal = np.expm1(y_pred_log)

y_pred_weighted = y_score_normal * df_score["outcome_damage_inc_proba"]
df_score["outcome_damage_amount"] = y_pred_weighted

df_score.to_csv("../score_cleaned.csv", index=False)

C:\Users\joppe\AppData\Local\Temp\ipykernel_36544\1952173042.py:16: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_score["outcome_damage_amount"] = y_pred_weighted


In [5]:
df_score_test = pd.read_csv("../score_cleaned.csv")
df_score_test.head()

,income_am,profit_last_am,profit_am,damage_am,damage_inc,crd_lim_rec,credit_use_ic,gluten_ic,lactose_ic,insurance_ic,...,score3_pos_missing,score3_neg_missing,score4_pos_missing,score4_neg_missing,score5_pos_missing,score5_neg_missing,outcome_damage_inc,outcome_damage_inc_proba,outcome_profit,outcome_damage_amount
0,5660.0,4320.0,8640.0,0.0,0.0,8000.0,0.0,0.0,1.0,0.0,...,1,1,1,1,1,1,0.0,0.454947,1530.043091,388.774865
1,3990.0,9.0,3450.0,0.0,0.0,12500.0,0.0,0.0,0.0,1.0,...,1,1,1,1,1,1,0.0,0.226547,2142.405305,168.066243
2,1158.0,82.0,4194.0,408.0,4.0,12000.0,0.0,0.0,0.0,1.0,...,0,0,0,0,1,1,0.0,0.338842,1506.288518,363.915308
3,2451.0,791.0,2119.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,1,1,1,1,1,1,0.0,0.224548,2139.112295,153.912524
4,946.0,222.0,2036.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,1,1,1,1,1,1,0.0,0.150515,1710.454505,79.695393
